# Level 3 — Core Numerical Methods Engine

## Scientific Objectives
1. Implement Root Finding (Bisection, Newton-Raphson, Secant) to calculate precise irrigation deficits considering non-linear drainage.
2. Implement Finite Differences (Forward, Backward, Central) for rate of moisture change.
3. Implement Numerical Integration (Trapezoidal, Simpson's) for cumulative deficits.
4. Implement Gaussian Elimination for multi-zone water allocation.

In [ ]:
import numpy as np
import pandas as pd

# 1. ROOT FINDING: Solving for required irrigation (I)
# Target soil = S_t + R - ET - D(I) + I
# Drainage is non-linear if the added water exceeds Field Capacity (FC)
# D(I) = coeff * max(0, S_t + I - FC). We want to find root of f(I) = 0.

def f_irrigation(I, S_t, target, R, ET, FC, coeff):
    drainage = coeff * max(0, S_t + I - FC)
    return (S_t + R - ET - drainage + I) - target

def bisection_method(func, a, b, tol=1e-5, max_iter=100):
    iters = 0
    if func(a) * func(b) >= 0:
        return None, iters # No bracket
    while (b - a) / 2.0 > tol and iters < max_iter:
        c = (a + b) / 2.0
        if func(c) == 0:
            break
        elif func(a) * func(c) < 0:
            b = c
        else:
            a = c
        iters += 1
    return (a + b) / 2.0, iters

def secant_method(func, x0, x1, tol=1e-5, max_iter=100):
    iters = 0
    for _ in range(max_iter):
        f_x0 = func(x0)
        f_x1 = func(x1)
        if abs(f_x1 - f_x0) < 1e-10: 
            break
        x2 = x1 - f_x1 * (x1 - x0) / (f_x1 - f_x0)
        if abs(x2 - x1) < tol:
            return x2, iters + 1
        x0, x1 = x1, x2
        iters += 1
    return x1, iters

# Example test scenario
S_t, target, R, ET, FC, coeff = 20.0, 35.0, 0.0, 5.0, 41.0, 0.18
f_val = lambda I: f_irrigation(I, S_t, target, R, ET, FC, coeff)

root_bisect, i_bisect = bisection_method(f_val, 0, 50)
root_secant, i_secant = secant_method(f_val, 0, 50)

print("=== ROOT FINDING: IRRIGATION REQUIREMENT ===")
print(f"Bisection: I = {root_bisect:.3f} mm (Iterations: {i_bisect})")
print(f"Secant:    I = {root_secant:.3f} mm (Iterations: {i_secant})")

In [ ]:
# 2. NUMERICAL DIFFERENTIATION
# Estimating rate of moisture change (dS/dt)
moisture = [33.2, 36.1, 33.7, 31.8, 33.3] # From Zone A first 5 days
dt = 1.0

print("\n=== FINITE DIFFERENCES (Day 2) ===")
idx = 2
fwd_diff = (moisture[idx+1] - moisture[idx]) / dt
bwd_diff = (moisture[idx] - moisture[idx-1]) / dt
cen_diff = (moisture[idx+1] - moisture[idx-1]) / (2*dt)

print(f"Forward diff:  {fwd_diff:.2f} %/day")
print(f"Backward diff: {bwd_diff:.2f} %/day")
print(f"Central diff:  {cen_diff:.2f} %/day")

In [ ]:
# 3. NUMERICAL INTEGRATION
def trapezoidal_rule(y, dx):
    return dx * (0.5 * y[0] + np.sum(y[1:-1]) + 0.5 * y[-1])

def simpson_rule(y, dx):
    n = len(y) - 1
    if n % 2 != 0:
        raise ValueError("Simpson's rule requires an even number of intervals.")
    res = y[0] + y[-1]
    for i in range(1, n, 2):
        res += 4 * y[i]
    for i in range(2, n-1, 2):
        res += 2 * y[i]
    return res * dx / 3.0

# Sample ET array over an even number of intervals (odd length array)
et_sample = np.array([4.1, 4.2, 4.5, 4.6, 4.4, 4.1, 3.8]) 
print("\n=== NUMERICAL INTEGRATION (Cumulative ET) ===")
print(f"Trapezoidal Rule: {trapezoidal_rule(et_sample, 1.0):.2f} mm")
print(f"Simpson's Rule:   {simpson_rule(et_sample, 1.0):.2f} mm")

In [ ]:
# 4. LINEAR SYSTEMS (Gaussian Elimination)
# Suppose we have 3 zones competing for a fixed 2000L water tank.
# Constraint 1: ZoneA + ZoneB + ZoneC = 2000
# Constraint 2: ZoneA needs twice as much as ZoneB: A - 2B = 0
# Constraint 3: ZoneC needs 1.5x of ZoneB: C - 1.5B = 0
# Matrix form Ax = b:
# [1,  1,   1] [A]   [2000]
# [1, -2,   0] [B] = [   0]
# [0, -1.5, 1] [C]   [   0]

def gaussian_elimination(A, b):
    n = len(b)
    M = np.hstack([A, b.reshape(-1, 1)]).astype(float)
    
    # Forward Elimination
    for i in range(n):
        for j in range(i+1, n):
            factor = M[j, i] / M[i, i]
            M[j, i:] -= factor * M[i, i:]
            
    # Back Substitution
    x = np.zeros(n)
    for i in range(n-1, -1, -1):
        x[i] = (M[i, -1] - np.dot(M[i, i+1:n], x[i+1:])) / M[i, i]
    return x

A = np.array([[1, 1, 1], [1, -2, 0], [0, -1.5, 1]])
b = np.array([2000, 0, 0])
allocation = gaussian_elimination(A, b)

print("\n=== GAUSSIAN ELIMINATION: WATER ALLOCATION ===")
print(f"Zone A gets: {allocation[0]:.1f} Liters")
print(f"Zone B gets: {allocation[1]:.1f} Liters")
print(f"Zone C gets: {allocation[2]:.1f} Liters")